# <font color='red'>
---
## <center> <font color='red'> Masters in Mathematical Finance
#### <center> <font color='red'> 2024 / 26
# <center> <font color='red'> Masters' Final Work
---
# <center> <font color='red'><font> Student: Petr Terletskiy </font>
### <center> <font color='red'><font> Number: l63023 </font>
---
##### <center>  <font color='red'><font> Project on Bitcoin's Volatility and Option Pricing Model Selection</font>

---

# Dependencies

In [ ]:
%pip install yfinance --quiet

Note: you may need to restart the kernel to use updated packages.


In [22]:
import pandas as pd
import numpy as np
from datetime import datetime
import requests

import yfinance as yf

import matplotlib.pyplot as plt
import seaborn as sns



import warnings
warnings.filterwarnings('ignore')

# Set up plotting style
#plt.style.use('seaborn-v0_8')
#sns.set_palette("husl")

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


# Data Retrieval

`btc_spot`: data, preço, volume

`btc_options`: data, strike, maturity, preço_compra, preço_venda, volume, etc.

`btc_on_chain`: data, transacoes_unicas, taxa_hash, dificuldade, volume_usd, idade_media_moedas, percentagem_inativas, fluxo_exchange, open_interest, funding_rate

### Spot Data

In [25]:
# 1. Download the data
ticker = 'BTC-USD'
btc_data_raw = yf.download(ticker, start="2011-12-31", end="2025-12-15")
print(f'{btc_data_raw.shape[0]} rows and {btc_data_raw.shape[1]} columns downloaded for {ticker} data.')

[*********************100%***********************]  1 of 1 completed

4107 rows and 5 columns downloaded for BTC-USD data.


In [ ]:
# 2. Feature Engineering
# We will create a new DataFrame to keep things clean.

# Simple Daily Return (percentage change)
btc_data_raw['daily_return'] = btc_data_raw['Close'].pct_change()

# Log Daily Return
btc_data_raw['log_daily_return'] = np.log(btc_data_raw['Close'] / btc_data_raw['Close'].shift(1))

# Intraday Range (absolute price difference)
btc_data_raw['intraday_range'] = btc_data_raw['High'] - btc_data_raw['Low']

# Normalized Intraday Range (as a percentage of the opening price)
btc_data_raw['norm_intraday_range'] = (btc_data_raw['High'] - btc_data_raw['Low']) / btc_data_raw['Open']

# 3. Construct the Final Dataset
# Select only the columns we need for the final dataset.
# We'll keep the original 'Close' and 'Volume' as they are fundamental.
final_columns = [
    'Close',
    'Volume',
    'daily_return',
    'log_daily_return',
    'intraday_range',
    'norm_intraday_range'
]

btc_spot = btc_data_raw[final_columns].copy()
btc_spot = btc_spot.reset_index()
btc_spot.columns = ['date', 'close_price', 'volume', 'daily_return', 'log_daily_return', 'intraday_range', 'norm_intraday_range']

# 4. Clean the Data
# The first row will have NaN (Not a Number) values because of the shift() operation.
# We remove it to have a clean dataset.
btc_spot.dropna(inplace=True)

# 5. Display the Result
print("--- Final Refined `btc_spot` Dataset ---")
btc_spot.head()

| Column Name | What It Is | Why It's Useful for Your Project |
| :--- | :--- | :--- |
| **`Close`** | The final trading price of BTC for that day. | This is your primary price reference. All returns are calculated from it. |
| **`Volume`** | The total amount of BTC traded during the day. | High volume often confirms the strength of a price move and is correlated with volatility. |
| **`daily_return`** | The percentage change from yesterday's close to today's close: `(Close_t / Close_{t-1}) - 1`. | Easy to interpret (e.g., 0.02 means a 2% gain). It's the basis for calculating **realized volatility** (e.g., the standard deviation of these returns over the last 30 days). |
| **`log_daily_return`** | The logarithmic return: `log(Close_t / Close_{t-1})`. | **This is the standard in quantitative finance.** Log returns are time-additive, which is a very useful mathematical property. They also tend to be more normally distributed than simple returns, which is an assumption in many models. |
| **`intraday_range`** | The absolute difference between the day's high and low prices: `High - Low`. | A direct, raw measure of how much the price fluctuated *within* the day. A large range signifies high intraday volatility and uncertainty. |
| **`norm_intraday_range`** | The intraday range as a percentage of the day's opening price: `(High - Low) / Open`. | This is a **superior feature** to the raw range. By normalizing, you can compare the intraday volatility across different price levels. A $500 range meant something different when BTC was at $20,000 versus when it was at $60,000. This feature adjusts for that. |

### Derivatives Data

### On-chain Data


#### Coinmetrics

In [35]:
!pip install coinmetrics.api_client --quiet

In [36]:
from coinmetrics.api_client import CoinMetricsClient

client = CoinMetricsClient()
print(client)

In [42]:
metrics = client.get_asset_metrics(assets='btc',
                                   metrics=[
                                       'PriceUSD',
                                       'AdrActCnt',
                                       'HashRate',
                                       'SplyCur',
                                       'TxCnt'],
                                   start_time = '2010-12-31',
                                   end_time='2025-12-15',
                                   frequency='1d')

print(metrics)

In [43]:
# Convert to DataFrame directly
df = metrics.to_dataframe()

# Convert timestamp to datetime if needed
df['time'] = pd.to_datetime(df['time'])

# Set 'time' as index
df.set_index('time', inplace=True)

# Display the df
df

,asset,PriceUSD,AdrActCnt,HashRate,SplyCur,TxCnt,SplyCur-status-time,SplyCur-status
time,,,,,,,,
2010-12-31 00:00:00+00:00,btc,0.3,876,0.116642,5020600.0,382,<NA>,<NA>
2011-01-01 00:00:00+00:00,btc,0.3,1071,0.130322,5029650.0,473,<NA>,<NA>
2011-01-02 00:00:00+00:00,btc,0.29997,1097,0.126002,5038400.0,418,<NA>,<NA>
2011-01-03 00:00:00+00:00,btc,0.295,1405,0.123832,5046200.0,997,<NA>,<NA>
2011-01-04 00:00:00+00:00,btc,0.29895,1111,0.114301,5053250.0,842,<NA>,<NA>
...,...,...,...,...,...,...,...,...
2025-12-11 00:00:00+00:00,btc,92623.719871,656012,1179627150.376248,19960716.785629,475726,<NA>,<NA>
2025-12-12 00:00:00+00:00,btc,90331.608309,761375,1068190149.670111,19961169.910622,411109,<NA>,<NA>
2025-12-13 00:00:00+00:00,btc,90245.267264,601306,1053456501.797791,19961616.785576,500012,<NA>,<NA>


In [48]:
# Display metrics df
list_asset_metrics = client.catalog_asset_metrics_v2(assets='btc')
btc_list = list_asset_metrics.to_dataframe()
btc_list

,asset,metric,frequency,min_time,max_time,min_height,max_height,min_hash,max_hash,community
0,btc,AdrActCnt,1d,2009-01-03 00:00:00+00:00,2025-12-26 00:00:00+00:00,None,None,None,None,True
1,btc,AdrBalCnt,1d,2009-01-03 00:00:00+00:00,2025-12-26 00:00:00+00:00,None,None,None,None,True
2,btc,AssetCompletionTime,1d,2009-01-03 00:00:00+00:00,2025-12-26 00:00:00+00:00,None,None,None,None,True
3,btc,AssetEODCompletionTime,1d,2009-01-03 00:00:00+00:00,2025-12-26 00:00:00+00:00,None,None,None,None,True
4,btc,BlkCnt,1d,2009-01-03 00:00:00+00:00,2025-12-26 00:00:00+00:00,None,None,None,None,True
5,btc,CapMVRVCur,1d,2010-07-18 00:00:00+00:00,2025-12-26 00:00:00+00:00,None,None,None,None,True
6,btc,CapMrktCurUSD,1d,2010-07-18 00:00:00+00:00,2025-12-26 00:00:00+00:00,None,None,None,None,True
7,btc,CapMrktEstUSD,1d,2019-06-22 00:00:00+00:00,2025-12-26 00:00:00+00:00,None,None,None,None,True
8,btc,FeeTotNtv,1d,2009-01-03 00:00:00+00:00,2025-12-26 00:00:00+00:00,None,None,None,None,True
9,btc,FlowInExNtv,1d,2009-01-03 00:00:00+00:00,2025-12-26 00:00:00+00:00,None,None,None,None,True


#### Blockchain.com

In [28]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import time

# Métricas disponíveis na Blockchain.com (gratuitas)
metrics = {
    'n-transactions': 'transactions',
    'n-unique-addresses': 'unique_addresses',
    'hash-rate': 'hash_rate',
    'difficulty': 'difficulty', 
    'miners-revenue': 'miners_revenue',
    'transaction-fees': 'transaction_fees',
    'market-price': 'market_price',
    'total-bitcoins': 'total_bitcoins',
    'mempool-size': 'mempool_size'
}

base_url = "https://api.blockchain.info/charts/"

# Calcular timespan em dias
start = datetime(2011, 1, 1)
end = datetime(2025, 11, 30)
timespan = (end - start).days

print("A obter dados da Blockchain.com...")
all_data = {}

for metric_key, metric_name in metrics.items():
    try:
        url = f"{base_url}{metric_key}"
        params = {
            'timespan': f'{timespan}days',
            'format': 'json',
            'sampled': 'false'
        }

        response = requests.get(url, params=params)

        if response.status_code == 200:
            data = response.json()
            values = data.get('values', [])

            # Converter timestamps para datas
            df_temp = pd.DataFrame(values)
            df_temp['date'] = pd.to_datetime(df_temp['x'], unit='s')
            df_temp = df_temp.rename(columns={'y': metric_name})
            df_temp = df_temp[['date', metric_name]]

            all_data[metric_name] = df_temp
            print(f"✓ {metric_name}: {len(df_temp)} pontos")
        else:
            print(f"✗ {metric_name}: erro {response.status_code}")

        time.sleep(0.5)  # Pausa para não sobrecarregar API

    except Exception as e:
        print(f"✗ {metric_name}: {e}")

# Combinar todos os dataframes
if all_data:
    df_final = None
    for metric_name, df_temp in all_data.items():
        if df_final is None:
            df_final = df_temp
        else:
            df_final = pd.merge(df_final, df_temp, on='date', how='outer')

    df_final = df_final.sort_values('date')

    # Apenas exibir informações, sem salvar CSV
    print(f"\n✓ Dados processados com sucesso!")
    print(f"Período: {df_final['date'].min()} até {df_final['date'].max()}")
    print(f"Dimensões: {len(df_final)} linhas × {len(df_final.columns)} colunas")
    print(f"Colunas disponíveis: {list(df_final.columns)}")
    
    # Mostrar estatísticas básicas
    print(f"\nEstatísticas básicas:")
    print(df_final.describe())
    
    # Mostrar primeiras e últimas linhas
    print(f"\nPrimeiras 5 linhas:")
    print(df_final.head())
    print(f"\nÚltimas 5 linhas:")
    print(df_final.tail())
    
    # Agora df_final está disponível para uso no seu projeto
    # Pode usá-lo diretamente para análise ou modelação
else:
    print("Nenhum dado foi obtido.")
    df_final = pd.DataFrame()  # Dataframe vazio para evitar erros

# O dataframe 'df_final' agora está disponível para uso no seu projeto
# Pode continuar a trabalhar com ele, por exemplo:
# - Criar features adicionais
# - Combinar com dados de outras fontes
# - Treinar modelos de machine learning

A obter dados da Blockchain.com...
✓ transactions: 5444 pontos
✓ unique_addresses: 5432 pontos
✓ hash_rate: 5444 pontos
✓ difficulty: 5444 pontos
✓ miners_revenue: 5447 pontos
✓ transaction_fees: 5444 pontos
✓ market_price: 5448 pontos
✓ total_bitcoins: 801526 pontos
✓ mempool_size: 334278 pontos

✓ Dados processados com sucesso!
Período: 2011-01-28 00:00:00 até 2025-12-27 22:00:00
Dimensões: 1137138 linhas × 10 colunas
Colunas disponíveis: ['date', 'transactions', 'unique_addresses', 'hash_rate', 'difficulty', 'miners_revenue', 'transaction_fees', 'market_price', 'total_bitcoins', 'mempool_size']

Estatísticas básicas:
                                date   transactions  unique_addresses  \
count                        1137138    5444.000000      5.432000e+03   
mean   2019-03-20 14:34:46.208970240  236560.735489      4.160045e+05   
min              2011-01-28 00:00:00     789.000000      9.890000e+02   
25%       2016-03-11 05:26:09.500000   79022.500000      1.730842e+05   
50%    

#### CryptoDataPy

In [50]:
!pip install cryptodatapy --quiet

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print("=" * 70)
print("📊 CRYPTODATAPY - EXTRACÇÃO DE DADOS DO BITCOIN (BTC)")
print("=" * 70)

# Inicializar o cliente CryptoDataPy
print("\n🔍 1. Inicializando CryptoDataPy e explorando fontes disponíveis...")

try:
    # Criar cliente
    client = cdp.Client()
    
    # Mostrar informações sobre o cliente
    print(f"✅ CryptoDataPy inicializado com sucesso!")
    print(f"   Versão: {cdp.__version__}")
    
    # 2. EXPLORAR DATASETS DISPONÍVEIS
    print("\n🔍 2. Explorando datasets disponíveis...")
    
    # Obter lista de datasets
    datasets = cdp.get_datasets()
    print(f"📁 Total de datasets disponíveis: {len(datasets)}")
    
    # Mostrar datasets relevantes para criptomoedas
    crypto_datasets = [d for d in datasets if any(x in d.lower() for x in ['coin', 'crypto', 'bitcoin', 'btc'])]
    print("\nDatasets relevantes para criptomoedas:")
    for i, dataset in enumerate(crypto_datasets[:10], 1):
        print(f"   {i}. {dataset}")
    
    # 3. EXPLORAR TICKERS DISPONÍVEIS
    print("\n🔍 3. Explorando tickers disponíveis para Bitcoin...")
    
    # Vamos focar nos datasets mais populares
    target_datasets = ['coinmetrics', 'glassnode', 'cryptocompare', 'binance']
    
    all_tickers = {}
    
    for dataset in target_datasets:
        if dataset in datasets:
            try:
                tickers = cdp.get_tickers(dataset)
                all_tickers[dataset] = tickers
                print(f"\n📊 {dataset.upper()}:")
                print(f"   Total de tickers: {len(tickers)}")
                
                # Filtrar tickers relacionados a Bitcoin
                btc_tickers = [t for t in tickers if 'btc' in t.lower() or 'bitcoin' in t.lower()]
                print(f"   Tickers de Bitcoin encontrados: {len(btc_tickers)}")
                
                if btc_tickers:
                    print(f"   Exemplos: {btc_tickers[:5]}")
                
            except Exception as e:
                print(f"   ❌ Erro ao obter tickers de {dataset}: {e}")
    
    # 4. EXPLORAR MÉTRICAS/CAMPOS DISPONÍVEIS
    print("\n🔍 4. Explorando métricas disponíveis para Bitcoin...")
    
    # Para cada dataset, verificar métricas disponíveis
    for dataset in ['coinmetrics', 'glassnode']:
        if dataset in datasets:
            try:
                fields = cdp.get_fields(dataset)
                print(f"\n📈 {dataset.upper()} - Métricas disponíveis:")
                print(f"   Total de métricas: {len(fields)}")
                
                # Categorizar métricas
                categories = {
                    'Preço & Mercado': ['price', 'market', 'cap', 'volume', 'return'],
                    'Rede & Blockchain': ['transaction', 'address', 'block', 'hash', 'difficulty', 'fee'],
                    'Mineração': ['miner', 'hashrate', 'reward', 'issuance'],
                    'Exchange': ['flow', 'exchange', 'balance', 'withdrawal', 'deposit'],
                    'Derivativos': ['future', 'option', 'funding', 'openinterest']
                }
                
                for category, keywords in categories.items():
                    cat_fields = []
                    for field in fields:
                        field_lower = field.lower()
                        if any(keyword in field_lower for keyword in keywords):
                            cat_fields.append(field)
                    
                    if cat_fields:
                        print(f"\n   {category} ({len(cat_fields)}):")
                        for field in cat_fields[:5]:  # Mostrar até 5
                            print(f"      • {field}")
                        if len(cat_fields) > 5:
                            print(f"      ... e mais {len(cat_fields) - 5}")
                
            except Exception as e:
                print(f"   ❌ Erro ao obter métricas de {dataset}: {e}")
    
    # 5. EXTRAIR DADOS DO BITCOIN
    print("\n" + "=" * 70)
    print("🚀 5. Extraindo dados do Bitcoin...")
    print("=" * 70)
    
    # Definir parâmetros de extração
    start_date = '2015-01-01'
    end_date = datetime.now().strftime('%Y-%m-%d')
    
    # Lista de métricas para extrair (baseado na disponibilidade)
    btc_metrics = {
        'coinmetrics': [
            'price_usd', 'volume_usd', 'marketcap_usd',
            'active_addresses', 'transaction_count', 'hashrate',
            'difficulty', 'fee_mean_usd', 'issuance_usd'
        ],
        'glassnode': [
            'price_usd_close', 'exchange_net_flow', 'miners_to_exchanges',
            'hash_rate_mean', 'difficulty_latest', 'fee_rate_mean'
        ],
        'cryptocompare': [
            'close', 'volumefrom', 'volumeto', 'high', 'low'
        ]
    }
    
    all_dataframes = {}
    
    # Extrair dados de cada fonte
    for source, metrics in btc_metrics.items():
        if source in datasets:
            print(f"\n📥 Extraindo dados de {source.upper()}...")
            
            try:
                # Tentar extrair dados
                df = cdp.get(
                    dataset=source,
                    tickers=['btc'],
                    fields=metrics,
                    start_date=start_date,
                    end_date=end_date,
                    freq='d'  # Frequência diária
                )
                
                if df is not None and not df.empty:
                    print(f"   ✅ Sucesso! {len(df)} registos obtidos")
                    print(f"   📊 Colunas: {list(df.columns)}")
                    
                    # Converter multi-index para formato mais simples se necessário
                    if isinstance(df.columns, pd.MultiIndex):
                        df.columns = ['_'.join(col).strip() for col in df.columns.values]
                    
                    # Guardar dataframe
                    all_dataframes[source] = df
                    
                    # Mostrar amostra
                    print(f"   📋 Primeiras linhas:")
                    print(df.head(3))
                    
                else:
                    print(f"   ⚠️  Nenhum dado retornado de {source}")
                    
            except Exception as e:
                print(f"   ❌ Erro ao extrair de {source}: {e}")
    
    # 6. COMBINAR DADOS DAS DIFERENTES FONTES
    print("\n" + "=" * 70)
    print("🔗 6. Combinando dados das diferentes fontes...")
    print("=" * 70)
    
    if all_dataframes:
        # Começar com o primeiro dataframe
        combined_df = None
        
        for source, df in all_dataframes.items():
            if combined_df is None:
                combined_df = df
            else:
                # Juntar dataframes pela data (index)
                try:
                    if isinstance(df.index, pd.DatetimeIndex):
                        combined_df = pd.merge(
                            combined_df, 
                            df, 
                            left_index=True, 
                            right_index=True, 
                            how='outer',
                            suffixes=('', f'_{source}')
                        )
                    else:
                        # Se não tiver DatetimeIndex, tentar juntar por uma coluna de data
                        date_cols = [col for col in df.columns if 'date' in col.lower() or 'time' in col.lower()]
                        if date_cols:
                            combined_df = pd.merge(combined_df, df, on=date_cols[0], how='outer')
                        
                except Exception as e:
                    print(f"   ⚠️  Não foi possível juntar dados de {source}: {e}")
        
        if combined_df is not None:
            # Ordenar por data
            if isinstance(combined_df.index, pd.DatetimeIndex):
                combined_df = combined_df.sort_index()
            else:
                # Tentar encontrar coluna de data
                date_cols = [col for col in combined_df.columns if 'date' in col.lower() or 'time' in col.lower()]
                if date_cols:
                    combined_df = combined_df.sort_values(date_cols[0])
            
            print(f"\n✅ Dados combinados com sucesso!")
            print(f"   📊 Dimensões: {combined_df.shape[0]} linhas × {combined_df.shape[1]} colunas")
            print(f"   📅 Período: {combined_df.index.min() if isinstance(combined_df.index, pd.DatetimeIndex) else 'N/A'} até {combined_df.index.max() if isinstance(combined_df.index, pd.DatetimeIndex) else 'N/A'}")
            
            # Mostrar estatísticas
            print(f"\n📈 Estatísticas básicas:")
            print(combined_df.describe())
            
            # Salvar dados combinados
            combined_df.to_csv('btc_cryptodatapy_combined.csv')
            print(f"\n💾 Dados combinados salvos como 'btc_cryptodatapy_combined.csv'")
            
            # 7. ANÁLISE EXPLORATÓRIA DOS DADOS
            print("\n" + "=" * 70)
            print("📊 7. Análise exploratória dos dados extraídos...")
            print("=" * 70)
            
            # Verificar valores nulos
            print(f"\n🔍 Valores nulos por coluna:")
            null_counts = combined_df.isnull().sum()
            for col, count in null_counts.items():
                if count > 0:
                    print(f"   {col}: {count} nulos ({count/len(combined_df)*100:.1f}%)")
            
            # Correlação entre variáveis principais (se tiver dados numéricos suficientes)
            numeric_cols = combined_df.select_dtypes(include=[np.number]).columns
            
            if len(numeric_cols) > 1:
                print(f"\n📊 Correlação entre principais variáveis:")
                
                # Selecionar até 10 variáveis numéricas para mostrar correlação
                if len(numeric_cols) > 10:
                    # Escolher variáveis com menos valores nulos
                    numeric_cols_subset = numeric_cols[:10]
                else:
                    numeric_cols_subset = numeric_cols
                
                corr_matrix = combined_df[numeric_cols_subset].corr()
                print(corr_matrix.head())
                
                # Recomendações baseadas nos dados
                print(f"\n🎯 RECOMENDAÇÕES para o seu projeto:")
                print(f"   1. Use {len(numeric_cols)} variáveis numéricas disponíveis")
                print(f"   2. Considere tratar {len(null_counts[null_counts > 0])} colunas com valores nulos")
                print(f"   3. Combine com dados da Blockchain.com e CoinMetrics")
                print(f"   4. Foque-se nas métricas com maior correlação com o preço")
    
    else:
        print("❌ Nenhum dado foi extraído com sucesso.")
        
    # 8. CÓDIGO ALTERNATIVO PARA EXTRAÇÃO DIRETA
    print("\n" + "=" * 70)
    print("💻 8. Código simplificado para extração direta:")
    print("=" * 70)
    
    simple_code = """
# CÓDIGO SIMPLIFICADO PARA EXTRAIR DADOS DO BTC COM CRYPTODATAPY

import cryptodatapy as cdp
import pandas as pd

# Extrair dados do Bitcoin
btc_data = cdp.get(
    dataset='coinmetrics',      # Fonte de dados
    tickers=['btc'],            # Ticker do Bitcoin
    fields=[
        'price_usd',           # Preço em USD
        'active_addresses',    # Endereços ativos
        'transaction_count',   # Número de transações
        'hashrate',            # Hash rate da rede
        'difficulty',          # Dificuldade de mineração
        'fee_mean_usd',        # Taxa média
        'marketcap_usd',       # Capitalização de mercado
        'volume_usd'           # Volume
    ],
    start_date='2015-01-01',   # Data inicial
    end_date='2024-12-31',     # Data final
    freq='d'                   # Frequência diária
)

# Converter para DataFrame
btc_df = pd.DataFrame(btc_data)

# Reorganizar colunas se for MultiIndex
if isinstance(btc_df.columns, pd.MultiIndex):
    btc_df.columns = ['_'.join(col).strip() for col in btc_df.columns.values]

# Mostrar resultados
print(f"Dados extraídos: {btc_df.shape}")
print(btc_df.head())
"""
    
    print(simple_code)
    
    # 9. TESTAR COM OUTRAS FONTES POPULARES
    print("\n" + "=" * 70)
    print("🧪 9. Testando extração de fontes adicionais...")
    print("=" * 70)
    
    # Testar com fontes específicas
    test_sources = ['binance', 'cryptocompare']
    
    for source in test_sources:
        if source in datasets:
            print(f"\nTestando {source.upper()}...")
            try:
                test_df = cdp.get(
                    dataset=source,
                    tickers=['BTCUSDT' if source == 'binance' else 'BTC'],
                    fields=['close', 'volume'],
                    start_date='2024-11-01',
                    end_date='2024-11-07',
                    freq='d'
                )
                
                if test_df is not None and not test_df.empty:
                    print(f"   ✅ Dados obtidos: {test_df.shape}")
                    # Verificar estrutura dos dados
                    print(f"   📊 Estrutura: {type(test_df)}")
                    if hasattr(test_df, 'columns'):
                        print(f"   🏷️  Colunas: {list(test_df.columns)[:5]}...")
                else:
                    print(f"   ⚠️  Sem dados retornados")
                    
            except Exception as e:
                print(f"   ❌ Erro: {e}")

except Exception as e:
    print(f"\n❌ Erro geral: {e}")
    print("\n💡 Soluções possíveis:")
    print("1. Verifique a instalação: pip install cryptodatapy --upgrade")
    print("2. Algumas fontes podem requerer chaves API (ex: Glassnode)")
    print("3. Consulte a documentação: https://cryptodatapy.readthedocs.io/")

print("\n" + "=" * 70)
print("🎯 PRÓXIMOS PASSOS PARA O SEU PROJETO:")
print("=" * 70)
print("""
1. Combine os dados da CryptoDataPy com os dados que já tens:
   • Blockchain.com (já tens)
   • CoinMetrics (se extraíres)

2. Estruture um pipeline de dados:
   btc_data = pd.merge(
       blockchain_df,
       coinmetrics_df,
       cryptodatapy_df,
       on='date',
       how='outer'
   )

3. Features importantes para previsão de BTC:
   • Preço histórico (lag features)
   • Volume e atividade de rede
   • Dados de mineração (hashrate, difficulty)
   • Fluxo de exchanges
   • Indicadores on-chain (NVT, MVRV)

4. Tratamento de dados:
   • Handle missing values
   • Normalize/scale features
   • Create lag features (1d, 7d, 30d returns)
   • Calculate technical indicators

5. Modelagem:
   • Split train/test (80/20 com time series)
   • Try models: LSTM, XGBoost, Random Forest
   • Use walk-forward validation for time series
""")

📊 CRYPTODATAPY - EXTRACÇÃO DE DADOS DO BITCOIN (BTC)

🔍 1. Inicializando CryptoDataPy e explorando fontes disponíveis...

❌ Erro geral: module 'cryptodatapy' has no attribute 'Client'

💡 Soluções possíveis:
1. Verifique a instalação: pip install cryptodatapy --upgrade
2. Algumas fontes podem requerer chaves API (ex: Glassnode)
3. Consulte a documentação: https://cryptodatapy.readthedocs.io/

🎯 PRÓXIMOS PASSOS PARA O SEU PROJETO:

1. Combine os dados da CryptoDataPy com os dados que já tens:
   • Blockchain.com (já tens)
   • CoinMetrics (se extraíres)

2. Estruture um pipeline de dados:
   btc_data = pd.merge(
       blockchain_df,
       coinmetrics_df,
       cryptodatapy_df,
       on='date',
       how='outer'
   )

3. Features importantes para previsão de BTC:
   • Preço histórico (lag features)
   • Volume e atividade de rede
   • Dados de mineração (hashrate, difficulty)
   • Fluxo de exchanges
   • Indicadores on-chain (NVT, MVRV)

4. Tratamento de dados:
   • Handle missing va